In [ ]:
import pandas as pd

data = pd.read_csv(
    "./../data/raw/shopify_sales_data.csv",
    dtype={"sku": str, "art_number": str},
)
for col in ["sold_at", "listed_at", "published_at"]:
    data[col] = pd.to_datetime(data[col], errors="coerce", utc=True)

print(len(data))

In [ ]:
data.head()

In [ ]:
# Checking coverage

for col in ["condition_grade", "art_number", "size", "colour", "brand", "measurements"]:
    filled = data[col].notna() & (data[col].astype(str).str.strip() != "")
    print(f"{col:18s} {filled.mean():6.1%}")

In [ ]:
no_prod = data["brand"].isna() | (data["brand"].astype(str).str.strip() == "")
print(no_prod.mean())                     # should be ~30.3%
print(data.loc[no_prod, "sold_at"].min(), "→", data.loc[no_prod, "sold_at"].max())
print(data.loc[no_prod, "title"].head(10))

In [ ]:
CORE_COLS = ["brand", "size", "colour", "condition_grade",
             "list_price", "listed_at", "published_at",
             "sold_at", "sold_price", "title", "sku"]

filled = pd.DataFrame({
    col: data[col].notna() & (data[col].astype(str).str.strip() != "")
    for col in CORE_COLS
})
complete = filled.all(axis=1)

print(f"complete (all core fields): {complete.sum()} rows ({complete.mean():.1%})")
print(f"of which also have measurements: "
      f"{(complete & (data['measurements'].fillna('') != '')).sum()}")

# What's blocking the incomplete rows — count missing fields per column:
print("\nmissing-field counts among incomplete rows:")
print((~filled[~complete]).sum().sort_values(ascending=False))

In [ ]:
# Checking range

CATEGORICAL_COLS = ["brand", "size", "colour", "condition_grade", "season", "product_name"]

for col in CATEGORICAL_COLS:
    vc = data[col].fillna("").astype(str).str.strip().value_counts()
    n_unique = len(vc)
    less_than_5 = (vc < 5).sum()
    print(f"\n=== {col} — {n_unique} unique values, {less_than_5} appear less than five times ===")
    print(vc.head(20))
    if n_unique > 20:
        print("  ... tail:", ", ".join(vc.tail(10).index))

In [ ]:
spike = data["listed_at"].dt.date.value_counts().head(3)
print(spike)  # how many rows share the import date?
data["clock_start"] = data["listed_at"]
# rows on the spike date get clock_start = NaT (untrusted)